In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
import torch
from transformers import AutoTokenizer, AutoModel 
from deutschland import klinikatlas
from deutschland.klinikatlas.api import default_api
from deutschland.klinikatlas.model.fileadmin_json_icd_codes_json_get200_response_inner import (
    FileadminJsonIcdCodesJsonGet200ResponseInner as klinikatlas_datatype
)
from transformers import XLMRobertaTokenizer, XLMRobertaModel


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [3]:
BASE_URL = "https://bundes-klinik-atlas.de"

configuration = klinikatlas.Configuration(
    host=BASE_URL
)

with klinikatlas.ApiClient(configuration) as api_client:
    api = default_api.DefaultApi(api_client)

    icd_data = api.fileadmin_json_icd_codes_json_get()

In [4]:
tokenizer = AutoTokenizer.from_pretrained("permediq/SapBERT-DE", use_fast=True)
model = AutoModel.from_pretrained("permediq/SapBERT-DE").to(device)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [5]:
def preprocess_input_string(input_str:str| list[str])->list[dict]:
    '''
    Takes a string or list of strings and then transforms it such that it can be used in german_medical_embedding. It is meant to be used on the 
    user search and not the data base or any larger set.  

    Args:
    input_str: String or list of strings. If the value is string it is space seperated while if it is a list of string it is seperated like by the list.

    Returns: 
    list_preprocessed_string: List of dictionaries all containing the key 'description' with value being the either the key seperated word or the
    different strings in the list. 
    '''
    input_str = input_str.split() if isinstance(input_str, str) else input_str
        
    list_preprocessed_string = []
    for value in input_str:
        list_preprocessed_string.append({'description':value})
    return list_preprocessed_string

In [6]:
def german_medical_embedding (data: list[klinikatlas_datatype]|list[dict]|str, 
                              bs:int, 
                              tokenizer:XLMRobertaTokenizer, 
                              model:XLMRobertaModel) -> torch.Tensor:
    '''
    Takes a list of strings then apply a pre trained model for medical semantic search to the list of strings returning a list of tensors.

    
    Args:
    data: String or list of stings being embeded.
    bs: Batch size 
    tokenizer, model: Part of applying the pre trained model to our model. 

    Returns:
    embedding: Embedded list of tensors
    
    '''
    data = preprocess_input_string(data) if isinstance(data, str) or isinstance(data[0], str) else data
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    embedding =[]
    for i in tqdm(np.arange(0, len(data), bs)):
        batch = data[i:i+bs]
        descriptions = [item["description"] for item in batch] 
        toks = tokenizer(
            descriptions,
            padding="max_length",
            max_length=40,
            truncation=True,
            return_tensors="pt"
        )
        
        tokse = {}
        for key,value in toks.items():
            tokse[key] = value.to(device)
        cls_rep = model(**tokse)[0][:,0,:] 
        embedding.append(cls_rep.cpu().detach())
    embedding = torch.cat(embedding)
    return embedding

In [7]:
def cos_sim(a, b):
    '''
    Takes two lists vectors and then returns the matrix of the angular difference. 

    Args:
    a,b: List of tensor vectors.

    Return: 
    Torch.mm: The matrix given by the angle difference of all values in 'a' to all values in 'b'
    '''
    a_norm = torch.nn.functional.normalize(a, p=2, dim=1)
    b_norm = torch.nn.functional.normalize(b, p=2, dim=1)
    return torch.mm(a_norm, b_norm.transpose(0, 1))

# cosine similarity of first entity with all the entities



In [8]:
def max_log(embedded_database: torch.Tensor, embedded_input: torch.Tensor)-> int:
    '''
    From the matrix given by cos_sim take the max of the logs summed of the ones corresponding to the same database embeddin. 
    
    Args:
    embedded_input: the input already embedded
    embedded_database: the database already embedded

    Return: 
    index: The index of the argmax of log summed angular difference from the input to the database
    '''
    similarity_tensor = cos_sim(embedded_database, embedded_input)
    log = torch.log(similarity_tensor).sum(dim=1)
    log = log.nan_to_num(nan = float('-inf'))
    index= log.argmax().item()
    return(index)

In [9]:
def max_nolog(embedded_database: torch.Tensor, embedded_input: torch.Tensor)-> int:
    '''
    From the matrix given by cos_sim take the sum of the ones corresponding to the same database embedding.
    
    Args:
    embedded_input: the input already embedded
    embedded_database: the database already embedded

    Return: 
    index: The index of the argmax of summed angular difference from the input to the database
    '''
    similarity_tensor = cos_sim(embedded_database, embedded_input)
    sums = similarity_tensor.sum(dim=1)
    index= sums.argmax().item()
    return(index)

In [10]:
bs = 32
all_embs = german_medical_embedding(icd_data, bs, tokenizer,model)


100%|██████████| 524/524 [00:18<00:00, 28.69it/s]


In [11]:
testing = cos_sim(all_embs[0].unsqueeze(0), all_embs)
print(testing)
print(np.argmax(testing))

tensor([[1.0000, 0.9283, 0.7229,  ..., 0.3239, 0.3501, 0.3125]])
tensor(0)


In [12]:
x='Typhus abdominalis und Paratyphus'
y=preprocess_input_string(x)
z = ['Sonstige', 'näher', 'bezeichnete', 'Karzinome', 'der', 'Leber']
Z= ['Sonstige näher bezeichnete Karzinome der Leber']
print(x)

this_embed = german_medical_embedding (x, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])
k= max_nolog(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

Typhus abdominalis und Paratyphus


100%|██████████| 1/1 [00:00<00:00, 43.00it/s]

3
Typhus abdominalis und Paratyphus
A01
3
Typhus abdominalis und Paratyphus
A01


In [13]:
print(y)
this_embed = german_medical_embedding (y, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])
k= max_nolog(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

[{'description': 'Typhus'}, {'description': 'abdominalis'}, {'description': 'und'}, {'description': 'Paratyphus'}]


100%|██████████| 1/1 [00:00<00:00, 52.73it/s]

3
Typhus abdominalis und Paratyphus
A01
3
Typhus abdominalis und Paratyphus
A01


In [14]:
print(z)
this_embed = german_medical_embedding (z, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])
k= max_nolog(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

['Sonstige', 'näher', 'bezeichnete', 'Karzinome', 'der', 'Leber']


100%|██████████| 1/1 [00:00<00:00, 45.30it/s]

7064
Gicht, nicht näher bezeichnet: Mehrere Lokalisationen
M10.90
7064
Gicht, nicht näher bezeichnet: Mehrere Lokalisationen
M10.90


In [15]:
print(Z)
this_embed = german_medical_embedding (Z, bs, tokenizer, model)
k= max_log(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])
k= max_nolog(all_embs, this_embed)
print(k)
print(icd_data[k]['description'])
print(icd_data[k]['icdcode'])

['Sonstige näher bezeichnete Karzinome der Leber']


100%|██████████| 1/1 [00:00<00:00, 29.03it/s]

1072
Sonstige näher bezeichnete Karzinome der Leber
C22.7
1072
Sonstige näher bezeichnete Karzinome der Leber
C22.7
